In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI

import os 
from dotenv import load_dotenv
# This function will load all the variables from .env file into the environment
# make them available to in the os.environ dictionary
load_dotenv()   

if os.environ.get("GOOGLE_API_KEY"):
    print ("GOOGLE_API_KEY found in the environment variables")
else:
    raise ValueError ("GOOGLE_API_KEY not found in the environment variables")


from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

# for the string out praser 
from langchain_core.output_parsers import StrOutputParser


llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

GOOGLE_API_KEY found in the environment variables


# **Chain with Parallel Chain**    

 in this prompt_template ask the LLM to summarize a movie or just ask LLM to write a quick summary about a movie

In [2]:
# TASK -1 [prompt]
Prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie summarizer"),
    ("human", " Please summarizer the movie in brief : {input}") # this will reduce my token because i mention how my ouptut i need .
])

In [3]:
# Task -2 [LLM]
llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)


In [4]:
# Task -3 [str Parser]

Str_parser = StrOutputParser()  


In [5]:
# Task -4 [custom Runnable]
from langchain_core.runnables import RunnableLambda
 
def dictionary_maker(text:str) -> dict:
    return {"text":text} # return a dictionary with the text as the value

dictionary_maker_runnable = RunnableLambda(dictionary_maker) # now i can use the dictionary_maker_runnable as a runnable as object to creat a chain 
# we use the dictionary_maker we cannot directly in the langchain chain
# to use a function we need a add wrapper on top of our function 
# so that chain in langchain will treat our function as a runnable 


# **For Parallel Chain we have two option**

the first option is to create dedicated chain for this task . Dedicated chain mean it will run like task 1 2 3 and all . like it will have it its own task and it will do everything on its own creating the chain dedicated chain 

The second option is simple and it is much more scalable and it is easy to manage as well . 

use case : 
        See if we have let's say just a few tasks it is easy create a dedicated chain  but if we have multiple tasks we can even create a function 

important to Remember : 
         we can also create a function for the entire chain as well. so  we can even create a function for the entire chain  

# **for diagram refer words:** 

 in that for **chain 1** we will use the normal chain method we will create all the tasks for the chain => **first option** 

 In that for **chain 2** we will not create the chain we will only create one function and  we will just write the code for every tasks. => **second option** 

# **Parallel Chain 1**

**Core** :In the above chain we ask the LLM to create movie summary next the chain 1 we will create a linkedln post for the movie and for the chain 2 we will create instragram post for movie

In [ ]:
# Task -1 [Prompt]

linkedin_prompt=ChatPromptTemplate.from_messages([
    ("system"," you are a linkedin post generator"),
    ("human","create a post for the following text for Linkedkin :{text}")
])
#Task -2 [LLM]
llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

#Task -3[Str praser]
str_praser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm_gemini |str_praser # instead of creating chain like this we 
# we create a function 

# **Parallel Chain 2** 

In [10]:
from langchain_core.runnables import RunnableParallel , RunnableLambda

In [9]:
def insta_chain(text:dict):
    text=text["text"]
    # Task -1 [Prompt]

    insta_prompt=ChatPromptTemplate.from_messages([
        ("system"," you are a instagram post generator"),
        ("human","create a post for the following text for instagram :{text}")
    ])
    #Task -2 [LLM]
    llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

    #Task -3[Str praser]
    str_praser = StrOutputParser()

    chain_insta = insta_prompt | llm_gemini | str_praser
    result = chain_insta.invoke(text)
    return result

# both concept of parallel chain ar same but small difference is that first we create chain and then we invoke it 
# with second option we do same but in function  format on top of it 
# it is easy to manage because you can import this function in other file and use it there and import this function as a utility function

# second Benefit : 
   #you can make the changes in between tasks because if you do not like we easy change the prompt or llm or parser 
   #if we create function like this you do not need those function like dictionary_marker because we can make the change in line 

insta_chain_runnnable = RunnableLambda(insta_chain) 
# RunnableLambda is a class that takes a function as an argument and returns a runnable object by that we can use it as a chain 

we need to finalize the final orchestrator . that means we need to now arrange task one , task two and task three , task four and then these two parallel chains with Parallel chain 1 and Parallel chain 2 .   

# **final orchestrator** 

In [12]:
final_chain = (
     Prompt_template |  
     llm_gemini | 
     str_praser |
     dictionary_maker_runnable |
     # here we going use a special function  called Runnable parallel
     # this function will run all the chains in parallel
     # After the dictionary_maker_runnable is done it will run the RunnableParallel say that you need to create two parallel things.
     # we also pass a list of chains(link) to the RunnableParallel
     #but the branches because in diagram there are two branches and we have built-in function branches becuase this will manage it well and best

     RunnableParallel(branches ={ "linkedin":chain_linkedin,"instagram":insta_chain_runnnable} )
     
)

In [13]:
final_chain.invoke("KGF")

{'branches': {'linkedin': "Okay, here are a few LinkedIn post options based on the KGF synopsis you provided, catering to different angles and audiences:\n\n**Option 1: Focusing on Story & Impact**\n\n```\nJust finished re-watching KGF: Chapters 1 & 2, and I'm still blown away! 🤯 This isn't just action; it's a powerful story of ambition, rebellion, and the human cost of power. From Rocky's rise in the Mumbai underworld to his reign over the Kolar Gold Fields, the journey is captivating. The film masterfully explores themes of oppression, hope, and the consequences of unchecked ambition. Highly recommend for anyone who appreciates a well-crafted narrative with stunning visuals and a compelling protagonist. #KGF #KGFChapter2 #IndianCinema #ActionMovies #Storytelling #MovieRecommendation #Power #Ambition #Rebellion\n```\n\n**Option 2: Focusing on Leadership & Strategy (More Business-Oriented)**\n\n```\nKGF: Chapters 1 & 2 offer some fascinating parallels to the world of leadership and str

final chain= you can also treat this chain as a runnable.
# **Chain as a Runnable**

now we want to extend the final chain and to connect with one more function and that function beautify the output. because we have the dictionary  here and we do not want to present it like and we just want to beautify it.

In [14]:
# Task -1 [beautify function ]

def beautify(final_response:dict)->dict: # we also return str but this are two different types of return type so we cannot use it
    linkedin_response = final_response['branches']["linkedin"]
    instagram_response = final_response['branches']["instagram"]
    return {
        "linkedin":linkedin_response,
        "instagram":instagram_response
    }   
beautify_runnable = RunnableLambda(beautify)

# Task -2 [final chain ]
# we have the final chain  already created
#final_chain 
# # we will create a runnable lambda for  this . we have two options i can even simply write final chain like this "final_chain"

#Beautified Chain

beautified_chain = final_chain | beautify_runnable #  now we can literally use the entire chain as my runnable . this is an by default behavior of langchain 

beautified_chain.invoke("Pushpa")



{'linkedin': 'Okay, here are a few LinkedIn post options based on your "Pushpa: The Rise" description, catering to different angles and audiences:\n\n**Option 1 (Business/Entrepreneurial Angle):**\n\n> Just finished watching "Pushpa: The Rise" and while it\'s a high-octane action film, it also offers some interesting insights into ambition and navigating a competitive landscape. Pushpa Raj\'s journey from a coolie to a powerful figure in a red sandalwood smuggling syndicate is a study in determination, albeit in a morally questionable context. It highlights the importance of understanding your market, building alliances (and knowing when to break them), and the relentless pursuit of your goals.  What are your thoughts on the portrayal of ambition in film and how it relates to the business world? #PushpaTheRise #IndianCinema #BusinessLessons #Entrepreneurship #Leadership #Motivation #FilmReview\n\n**Option 2 (General Interest/Culture):**\n\n> "Pushpa: The Rise" is a wild ride! This Indi

Important to Remember : chains are also runnable by default 

let say this a chain we built ,now this entire chain is runnable  and this nature of it 

the above sentence means you can just use this entire chain as your runnable and we  can connect it to the next task as beautify task so the property of it is whatever we create in the chain it is a runnable by default so we can literally write # final chain 

# **we need to explicitly convert your python function as Runnable but if you are creating a chain we don't need to convert into runnable . That is runnable by default .**